# Tarea DataFrames PySpark - Actividad 3
## Procesamiento de Grandes Bases de Datos

In [4]:
# Importar librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import shutil
import glob

In [5]:
# Crear sesión de Spark
spark = SparkSession.builder \
    .appName("Actividad3_DataFrames") \
    .getOrCreate()

In [3]:
# Función para eliminar directorio si existe
def limpiar_directorio(ruta):
    if os.path.exists(ruta):
        shutil.rmtree(ruta)
        print(f"Directorio {ruta} eliminado")

## 1. Cargar los datasets

In [4]:
# Cargar stocks_price_final.csv
df_stocks = spark.read.csv('stocks_price_final.csv', header=True, inferSchema=True)
print("Dataset de Stocks Prices cargado")
df_stocks.show(5)

Dataset de Stocks Prices cargado
+---+------+----------+---------+---------+---------+---------+-------+---------+----------+-------------+--------------------+--------+
|_c0|symbol|      date|     open|     high|      low|    close| volume| adjusted|market.cap|       sector|            industry|exchange|
+---+------+----------+---------+---------+---------+---------+-------+---------+----------+-------------+--------------------+--------+
|  1|   TXG|2019-09-12|       54|       58|       51|    52.75|7326300|    52.75|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  2|   TXG|2019-09-13|    52.75|   54.355|49.150002|    52.27|1025200|    52.27|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  3|   TXG|2019-09-16|52.450001|       56|52.009998|55.200001| 269900|55.200001|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  4|   TXG|2019-09-17|56.209999|60.900002|   55.423|56.779999| 602800|56.779999|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  5|   

In [5]:
# Cargar WorldCupPlayers.csv
df_players = spark.read.csv('WorldCupPlayers.csv', header=True, inferSchema=True)
print("Dataset de World Cup Players cargado")
df_players.show(5)

Dataset de World Cup Players cargado
+-------+-------+-------------+-------------------+-------+------------+----------------+--------+-----+
|RoundID|MatchID|Team Initials|         Coach Name|Line-up|Shirt Number|     Player Name|Position|Event|
+-------+-------+-------------+-------------------+-------+------------+----------------+--------+-----+
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0|     Alex THEPOT|      GK| NULL|
|    201|   1096|          MEX|   LUQUE Juan (MEX)|      S|           0| Oscar BONFIGLIO|      GK| NULL|
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0|Marcel LANGILLER|    NULL| G40'|
|    201|   1096|          MEX|   LUQUE Juan (MEX)|      S|           0|    Juan CARRENO|    NULL| G70'|
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0| Ernest LIBERATI|    NULL| NULL|
+-------+-------+-------------+-------------------+-------+------------+----------------+--------+-----+
only showing top 5

## 2. Exploración de datos - Stocks Prices

In [6]:
# Esquema del dataset de stocks
print("Esquema de stocks_price_final:")
df_stocks.printSchema()

Esquema de stocks_price_final:
root
 |-- _c0: integer (nullable = true)
 |-- symbol: string (nullable = true)
 |-- date: date (nullable = true)
 |-- open: string (nullable = true)
 |-- high: string (nullable = true)
 |-- low: string (nullable = true)
 |-- close: string (nullable = true)
 |-- volume: string (nullable = true)
 |-- adjusted: string (nullable = true)
 |-- market.cap: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- exchange: string (nullable = true)



In [7]:
# Número de registros y columnas
print(f"Número de registros: {df_stocks.count()}")
print(f"Número de columnas: {len(df_stocks.columns)}")

Número de registros: 1729034
Número de columnas: 13


## 3. Análisis - Stocks Prices

In [9]:
# Seleccionar columnas específicas
df_stocks.select('symbol', 'date', 'close', 'volume').show(10)

+------+----------+---------+-------+
|symbol|      date|    close| volume|
+------+----------+---------+-------+
|   TXG|2019-09-12|    52.75|7326300|
|   TXG|2019-09-13|    52.27|1025200|
|   TXG|2019-09-16|55.200001| 269900|
|   TXG|2019-09-17|56.779999| 602800|
|   TXG|2019-09-18|       62|1589600|
|   TXG|2019-09-19|61.119999| 425200|
|   TXG|2019-09-20|     60.5| 392000|
|   TXG|2019-09-23|60.330002| 137200|
|   TXG|2019-09-24|54.299999| 713800|
|   TXG|2019-09-25|52.759998| 261200|
+------+----------+---------+-------+
only showing top 10 rows



In [10]:
# Filtrar registros donde el precio de cierre sea mayor a 55
df_stocks.filter(col('close') > 55).show(10)

+---+------+----------+---------+---------+---------+---------+-------+---------+----------+-------------+--------------------+--------+
|_c0|symbol|      date|     open|     high|      low|    close| volume| adjusted|market.cap|       sector|            industry|exchange|
+---+------+----------+---------+---------+---------+---------+-------+---------+----------+-------------+--------------------+--------+
|  4|   TXG|2019-09-17|56.209999|60.900002|   55.423|56.779999| 602800|56.779999|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  5|   TXG|2019-09-18|56.849998|    62.27|55.650002|       62|1589600|       62|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  6|   TXG|2019-09-19|62.810001|   63.375|61.029999|61.119999| 425200|61.119999|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  7|   TXG|2019-09-20|61.709999|62.419998|    59.82|     60.5| 392000|     60.5|    $9.31B|Capital Goods|Biotechnology: La...|  NASDAQ|
|  8|   TXG|2019-09-23|60.220001|61.48500

In [11]:
# Ordenar por volumen de manera descendente
df_stocks.orderBy(col('volume').desc()).show(10)

+-------+------+----------+----+----+---+-----+------+--------+----------+-------------+--------------------+--------+
|    _c0|symbol|      date|open|high|low|close|volume|adjusted|market.cap|       sector|            industry|exchange|
+-------+------+----------+----+----+---+-----+------+--------+----------+-------------+--------------------+--------+
|1688838|   VRT|2019-02-27|  NA|  NA| NA|   NA|    NA|      NA|    $4.52B|Capital Goods|Industrial Machin...|    NYSE|
| 128387|  BFYT|2020-02-04|  NA|  NA| NA|   NA|    NA|      NA|  $441.77M|      Finance|  Specialty Insurers|  NASDAQ|
|1338261|   HWM|2020-02-06|  NA|  NA| NA|   NA|    NA|      NA|    $7.04B|Capital Goods|  Metal Fabrications|    NYSE|
| 128397|  BFYT|2020-02-19|  NA|  NA| NA|   NA|    NA|      NA|  $441.77M|      Finance|  Specialty Insurers|  NASDAQ|
| 391568|  GSMG|2020-02-05|  NA|  NA| NA|   NA|    NA|      NA|   $211.3M|   Technology|        EDP Services|  NASDAQ|
| 128388|  BFYT|2020-02-05|  NA|  NA| NA|   NA| 

In [12]:
# Calcular precio promedio de cierre por símbolo
df_stocks.groupBy('symbol').agg(
    avg('close').alias('precio_cierre_promedio'),
    sum('volume').alias('volumen_total')
).show()

+------+----------------------+-------------+
|symbol|precio_cierre_promedio|volumen_total|
+------+----------------------+-------------+
|  ALXN|     112.7071684489796|   7.779176E8|
|  ABMD|    228.89303535204076|   2.647128E8|
|   APM|    13.568482153061222|    8675800.0|
|  ARAY|    3.2255357142857135|   2.594767E8|
|  AMTX|    0.8796020408163266|   1.192674E8|
|   API|     50.22333322222223|    3.03322E7|
|  ARTW|    2.0451530612244917|    2.55165E7|
|  ADIL|    2.2348469387755094|    8.50632E7|
|  AVNW|    13.535816331632644|    7285000.0|
|  ALDX|     5.854834183673468|   1.916388E8|
|  APYX|     6.340624999999997|    6.69967E7|
|   CAR|     28.70397954081632|  1.0198641E9|
|  ATCX|      9.36369907407408|    9849900.0|
|  AHCO|    12.391104075144522|    4.85342E7|
|  AMRK|    12.459028033163266|    6024600.0|
|  AVXL|    3.1053443877551015|   2.430759E8|
|  AVEO|     7.375484701530607|   3.769877E8|
|  ASYS|     5.464566326530616|    2.20939E7|
|  AVDL|     4.837576530612244|   

In [13]:
# Agregar columna calculada: rango diario (high - low)
df_stocks_con_rango = df_stocks.withColumn('rango_diario', col('high') - col('low'))
df_stocks_con_rango.select('symbol', 'date', 'high', 'low', 'rango_diario').show(10)

+------+----------+---------+---------+------------------+
|symbol|      date|     high|      low|      rango_diario|
+------+----------+---------+---------+------------------+
|   TXG|2019-09-12|       58|       51|               7.0|
|   TXG|2019-09-13|   54.355|49.150002| 5.204997999999996|
|   TXG|2019-09-16|       56|52.009998| 3.990001999999997|
|   TXG|2019-09-17|60.900002|   55.423| 5.477001999999999|
|   TXG|2019-09-18|    62.27|55.650002|6.6199980000000025|
|   TXG|2019-09-19|   63.375|61.029999|2.3450010000000034|
|   TXG|2019-09-20|62.419998|    59.82|2.5999979999999994|
|   TXG|2019-09-23|61.485001|59.939999|1.5450019999999967|
|   TXG|2019-09-24|       61|       54|               7.0|
|   TXG|2019-09-25|55.880001|   52.563|3.3170009999999976|
+------+----------+---------+---------+------------------+
only showing top 10 rows



## 4. Exploración de datos - World Cup Players

In [14]:
# Esquema del dataset de jugadores
print("Esquema de WorldCupPlayers:")
df_players.printSchema()

Esquema de WorldCupPlayers:
root
 |-- RoundID: integer (nullable = true)
 |-- MatchID: integer (nullable = true)
 |-- Team Initials: string (nullable = true)
 |-- Coach Name: string (nullable = true)
 |-- Line-up: string (nullable = true)
 |-- Shirt Number: integer (nullable = true)
 |-- Player Name: string (nullable = true)
 |-- Position: string (nullable = true)
 |-- Event: string (nullable = true)



In [15]:
# Número de registros y columnas
print(f"Número de registros: {df_players.count()}")
print(f"Número de columnas: {len(df_players.columns)}")

Número de registros: 37784
Número de columnas: 9


In [16]:
# Ver valores únicos de Team Initials
df_players.select('Team Initials').distinct().show(50)

+-------------+
|Team Initials|
+-------------+
|          POL|
|          JAM|
|          BRA|
|          CUB|
|          FRA|
|          ALG|
|          BOL|
|          RSA|
|          ITA|
|          UKR|
|          CMR|
|          SCG|
|          GHA|
|          SEN|
|          TOG|
|          TRI|
|          TCH|
|          AUS|
|          MEX|
|          PAR|
|          GDR|
|          SVK|
|          HUN|
|          ENG|
|          NZL|
|          NOR|
|          IRQ|
|          SCO|
|          POR|
|          DEN|
|          ZAI|
|          URU|
|          ANG|
|          BIH|
|          NED|
|          PER|
|          INH|
|          TUR|
|          NIR|
|          AUT|
|          USA|
|          MAR|
|          CIV|
|          KOR|
|          PRK|
|          GER|
|          ISR|
|          TUN|
|          IRN|
|          YUG|
+-------------+
only showing top 50 rows



## 5. Análisis - World Cup Players

In [17]:
# Contar jugadores por equipo
df_players.groupBy('Team Initials').count().orderBy(col('count').desc()).show(20)

+-------------+-----+
|Team Initials|count|
+-------------+-----+
|          BRA| 2403|
|          ITA| 1843|
|          ARG| 1807|
|          ENG| 1378|
|          FRG| 1364|
|          FRA| 1344|
|          ESP| 1317|
|          NED| 1210|
|          MEX| 1190|
|          URU| 1159|
|          GER| 1088|
|          SWE|  992|
|          BEL|  944|
|          YUG|  799|
|          SUI|  753|
|          CHI|  748|
|          USA|  729|
|          HUN|  704|
|          KOR|  695|
|          POL|  688|
+-------------+-----+
only showing top 20 rows



In [18]:
# Contar jugadores por posición
df_players.groupBy('Position').count().orderBy(col('count').desc()).show()

+--------+-----+
|Position|count|
+--------+-----+
|    NULL|33641|
|      GK| 2441|
|       C| 1510|
|     GKC|  192|
+--------+-----+



In [19]:
# Filtrar jugadores de FRA (Francia)
df_players.filter(col('Team Initials') == 'FRA').show(20)

+-------+-------+-------------+-------------------+-------+------------+-----------------+--------+---------+
|RoundID|MatchID|Team Initials|         Coach Name|Line-up|Shirt Number|      Player Name|Position|    Event|
+-------+-------+-------------+-------------------+-------+------------+-----------------+--------+---------+
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0|      Alex THEPOT|      GK|     NULL|
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0| Marcel LANGILLER|    NULL|     G40'|
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0|  Ernest LIBERATI|    NULL|     NULL|
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0|  Andre MASCHINOT|    NULL|G43' G87'|
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0|  Etienne MATTLER|    NULL|     NULL|
|    201|   1096|          FRA|CAUDRON Raoul (FRA)|      S|           0|     Marcel PINEL|    NULL|     NULL|
|    201| 

In [20]:
# Filtrar jugadores de MEX (México)
df_players.filter(col('Team Initials') == 'MEX').show(20)

+-------+-------+-------------+----------------+-------+------------+--------------------+--------+-----+
|RoundID|MatchID|Team Initials|      Coach Name|Line-up|Shirt Number|         Player Name|Position|Event|
+-------+-------+-------------+----------------+-------+------------+--------------------+--------+-----+
|    201|   1096|          MEX|LUQUE Juan (MEX)|      S|           0|     Oscar BONFIGLIO|      GK| NULL|
|    201|   1096|          MEX|LUQUE Juan (MEX)|      S|           0|        Juan CARRENO|    NULL| G70'|
|    201|   1096|          MEX|LUQUE Juan (MEX)|      S|           0|        Rafael GARZA|       C| NULL|
|    201|   1096|          MEX|LUQUE Juan (MEX)|      S|           0|       Hilario LOPEZ|    NULL| NULL|
|    201|   1096|          MEX|LUQUE Juan (MEX)|      S|           0|      Dionisio MEJIA|    NULL| NULL|
|    201|   1096|          MEX|LUQUE Juan (MEX)|      S|           0|        Felipe ROSAS|    NULL| NULL|
|    201|   1096|          MEX|LUQUE Juan (MEX

In [21]:
# Contar jugadores por RoundID
df_players.groupBy('RoundID').count().orderBy('RoundID').show()

+-------+-----+
|RoundID|count|
+-------+-----+
|    201|  566|
|    202|   77|
|    204|  321|
|    206|  373|
|    208|  666|
|    209|  252|
|    211|  788|
|    212|  176|
|    220| 1188|
|    221|  176|
|    231| 1056|
|    232|  176|
|    238| 1056|
|    239|  176|
|    250| 1047|
|    251|  176|
|    262| 1056|
|    263|  528|
|    264|   44|
|    278| 1056|
+-------+-----+
only showing top 20 rows



In [22]:
# Jugadores que participaron en eventos (goles, etc.)
df_players.filter(col('Event').isNotNull()).show(20)

+-------+-------+-------------+--------------------+-------+------------+-------------------+--------+---------+
|RoundID|MatchID|Team Initials|          Coach Name|Line-up|Shirt Number|        Player Name|Position|    Event|
+-------+-------+-------------+--------------------+-------+------------+-------------------+--------+---------+
|    201|   1096|          FRA| CAUDRON Raoul (FRA)|      S|           0|   Marcel LANGILLER|    NULL|     G40'|
|    201|   1096|          MEX|    LUQUE Juan (MEX)|      S|           0|       Juan CARRENO|    NULL|     G70'|
|    201|   1096|          FRA| CAUDRON Raoul (FRA)|      S|           0|    Andre MASCHINOT|    NULL|G43' G87'|
|    201|   1096|          FRA| CAUDRON Raoul (FRA)|      S|           0|     Lucien LAURENT|    NULL|     G19'|
|    201|   1090|          USA|    MILLAR Bob (USA)|      S|           0|         Tom FLORIE|       C|     G45'|
|    201|   1090|          USA|    MILLAR Bob (USA)|      S|           0|        Bart McGHEE|   

## 6. Análisis combinado

In [23]:
# Crear tabla temporal para stocks
df_stocks.createOrReplaceTempView('stocks')

# Consulta SQL: precio máximo y mínimo por fecha
spark.sql("""
    SELECT date, 
           MAX(high) as precio_max, 
           MIN(low) as precio_min
    FROM stocks
    GROUP BY date
    ORDER BY date
""").show(10)

+----------+----------+----------+
|      date|precio_max|precio_min|
+----------+----------+----------+
|2019-01-02| 99.779999|      0.26|
|2019-01-03| 99.720001|      0.25|
|2019-01-04|        NA|      0.26|
|2019-01-07| 99.910004|     0.275|
|2019-01-08| 99.989998|     0.295|
|2019-01-09| 99.980003|      0.27|
|2019-01-10| 99.959999|      0.27|
|2019-01-11|        NA|      0.26|
|2019-01-14| 99.650002|     0.258|
|2019-01-15| 99.940002|     0.258|
+----------+----------+----------+
only showing top 10 rows



In [24]:
# Crear tabla temporal para jugadores
df_players.createOrReplaceTempView('players')

# Consulta SQL: jugadores por equipo y entrenador
spark.sql("""
    SELECT `Team Initials`, 
           `Coach Name`, 
           COUNT(*) as num_jugadores
    FROM players
    GROUP BY `Team Initials`, `Coach Name`
    ORDER BY num_jugadores DESC
""").show(20)

+-------------+--------------------+-------------+
|Team Initials|          Coach Name|num_jugadores|
+-------------+--------------------+-------------+
|          FRG| SCHOEN Helmut (FRG)|          550|
|          BRA| ZAGALLO Mario (BRA)|          440|
|          GER|  LOEW Joachim (GER)|          414|
|          BRA|SCOLARI Luiz Feli...|          414|
|          ITA|  BEARZOT Enzo (ITA)|          396|
|          URU| TABAREZ Oscar (URU)|          364|
|          FRG|HERBERGER Sepp (FRG)|          352|
|          BEL|      THYS Guy (BEL)|          351|
|          HUN|  BAROTI Lajos (HUN)|          330|
|          ARG|BILARDO Carlos (ARG)|          313|
|          FRG|BECKENBAUER Franz...|          308|
|          ENG|WINTERBOTTOM Walt...|          305|
|          URS|KACHALIN Gavril (...|          286|
|          BRA|PARREIRA Carlos A...|          269|
|          ENG|  ROBSON Bobby (ENG)|          264|
|          ARG|MENOTTI Cesar Lui...|          264|
|          ARG|SABELLA Alejandr

In [6]:
# Detener sesión de Spark
spark.stop()